In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd #to store data as csv or excel format
import time
from datetime import date
import re

WRITE_FILE_PATH = r'c:\Interest_rate_files'
CHROME_DRIVER_PATH = 'c:\\Interest_rate_files\\chromedriver-win64\\chromedriver.exe'
URL = "https://www.yesbank.in/personal-banking/yes-individual/deposits/fixed-deposit" 

date1 = (date.today()).strftime('%d%m%Y') #Get today's date and convert it to to dd/mm/yy format
file_name= WRITE_FILE_PATH + '\Interest_rate_' + date1 + '.csv' #Set directory path to write file

# Setup Chrome options
chrome_options = Options()
chrome_options.add_argument("--headless")  # Run in headless mode
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

# Provide the path to your ChromeDriver
service = Service(CHROME_DRIVER_PATH)

# Initialize the WebDriver
driver = webdriver.Chrome(executable_path=CHROME_DRIVER_PATH)

try:
    # Open the webpage
    driver.get(URL)

    # Wait until the content is loaded
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))

    # Give it extra time if needed
    time.sleep(5)

    # Get the page source after rendering JavaScript
    page_content = driver.page_source

    # You can parse this with BeautifulSoup if needed
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(page_content, 'html.parser')
    table=soup.find("table")
    rows=table.find_all("tr")
    
    print("Today's date =",date1)
    print("Interest rate file name = ",file_name)
    
    list_of_interest_rates = []
    tax_saver_row = []
    row_count = 0
    for i in rows[3:]:  #Skip heading
        data = i.find_all("td")
        row=['Yes Bank']
        row.extend([tr.text for tr in data])
        #print("row=",row)
        for i in range(0,6):
            #row[i] = ' '.join(row[i].strip().replace('\n', '').split())
            row[i] = ' '.join(re.sub(r'[\n@$#*\xa0]', '', row[i].strip()).split())
            #row[i] = re.sub(r'[\n@$#*\xa0]', '', row[i]).strip()
            # Normalize spaces
            #row[i] = ' '.join(row[i].split())
        if row_count == 13: 
            tax_saver_row = ['Yes Bank', 'Tax-saver Fixed Deposit', row[2], row[3], row[4], row[5]]
            list_of_interest_rates.append(tax_saver_row)
        list_of_interest_rates.append(row)
        row_count += 1
            
    #print(list_of_interest_rates)
    pd.set_option('display.max_columns', None)    
    int_rate_df=pd.DataFrame(list_of_interest_rates,columns=['bank_name','tenure','gen_rate','annualised_gen_rate',
                            'sr_rate','annualised_sr_rate'])
    print(int_rate_df)
    int_rate_df.to_csv(file_name, mode='a', header=False, index=False)
    print("\nYes Bank = Success. Number of rows added = ", len(list_of_interest_rates))
finally:
    # Close the driver
    driver.quit()

Today's date = 19112024
Interest rate file name =  c:\Interest_rate_files\Interest_rate_v2_19112024.csv
row= ['Yes Bank', '\n7 days to 14 days\n', '\n3.25%\n', '\n3.25%\n', '\n3.75%\n', '\n3.75%\n']
row= ['Yes Bank', '\n15 days to 45 days\n', '\n3.70%\n', '\n3.70%\n', '\n4.20%\n', '\n4.20%\n']
row= ['Yes Bank', '\n46 days to 90 days\n', '\n4.10%\n', '\n4.10%\n', '\n4.60%\n', '\n4.60%\n']
row= ['Yes Bank', '\n91 days to 120 days\n', '\n4.75%\n', '\n4.75%\n', '\n5.25%\n', '\n5.25%\n']
row= ['Yes Bank', '\n121 days to 180 days\n', '\n5.00%\n', '\n5.00%\n', '\n5.50%\n', '\n5.50%\n']
row= ['Yes Bank', '\n181 days to 271 days\n', '\n6.10%\n', '\n6.24%\n', '\n6.60%\n', '\n6.77%\n']
row= ['Yes Bank', '\n272 days to < 1 year\n', '\n6.35%\n', '\n6.50%\n', '\n6.85%\n', '\n7.03%\n']
row= ['Yes Bank', '\n1 year\xa0\n', '\n7.25%\n', '\n7.45%\n', '\n7.75%\n', '\n7.98%\n']
row= ['Yes Bank', '\n1 year 1 day to < 18 months\xa0\n', '\n7.50%\n', '\n7.71%\n', '\n8.00%\n', '\n8.24%\n']
row= ['Yes Bank', '\n